In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(GEOquery)
quiet_library(hise)
quiet_library(purrr)
quiet_library(dplyr)

Warning message:
“package ‘GEOquery’ was built under R version 4.4.2”
Warning message:
“package ‘Biobase’ was built under R version 4.4.2”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.4.2”
Warning message:
“package ‘purrr’ was built under R version 4.4.2”


In [2]:
quiet_library(reticulate)
use_python("/home/workspace/environment/minimal/bin/python")
anndata <- import("anndata")

Warning message:
“package ‘reticulate’ was built under R version 4.4.3”


In [3]:
if(!dir.exists("output")) {
    dir.create("output")
}

### Gene Metadata

BRI used UCSC hg19 (GRCh37) for alignment.

In [4]:
ensembl_url <- "https://ftp.ensembl.org/pub/grch37/current/gtf/homo_sapiens/Homo_sapiens.GRCh37.87.chr_patch_hapl_scaff.gtf.gz"
download.file(ensembl_url, "Homo_sapiens.GRCh37.87.chr_patch_hapl_scaff.gtf.gz")
system("gunzip Homo_sapiens.GRCh37.87.chr_patch_hapl_scaff.gtf.gz")

In [5]:
gtf <- read.csv(
    "Homo_sapiens.GRCh37.87.chr_patch_hapl_scaff.gtf", 
    sep = "\t", 
    skip = 5,
    header = FALSE
)

In [6]:
gtf_names <- c("seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attributes")
names(gtf) <- gtf_names

In [7]:
nrow(gtf)

[1] 2856446

In [8]:
gene_meta <- gtf %>%
  mutate(id = sub("gene_id ([^;]+);.+","\\1", attributes),
         name = ifelse(
             grepl("gene_name", attributes),
             sub(".+gene_name ([^;]+);.+","\\1", attributes),
             id
         )) %>%
  dplyr::select(id, name) %>%
  base::unique()

In [9]:
nrow(gene_meta)

[1] 63677

### Sample Metadata

In [10]:
meta_uuid <- 'd82c5c42-ae5f-4e67-956e-cd3b7bf88105'
meta_file <- cacheFiles(list(meta_uuid))
sample_meta <- read.csv(meta_file)

[1] "downloading fileID d82c5c42-ae5f-4e67-956e-cd3b7bf88105"


Select relevant columns to retain

In [11]:
sample_meta <- sample_meta %>%
  mutate(sample.drawYear = sub("-.+", "", sample.drawDate)) %>%
  dplyr::select(-sample.drawDate, -sample.bridgingControl, -subject.id, -sample.reference, -sample.id, -sample.diseaseStatesRecordedAtVisit, -sample.visitDetails) %>%
  dplyr::select(starts_with("cohort"), starts_with("subject"), starts_with("sample"))

Prepare for combination with the GEO data by making a conversion column for timepoints

In [12]:
timepoint_conversion <- data.frame(
    geo.timepoint = c(
        "Y1 Flu Day 0", "Y1 Flu Day 7-9", "Y1 Flu Day 80-100",
        "Y2 Flu Day 0", "Y2 Flu Day 7-9", "Y2 Flu Day 80-100",
        "Non-Flu Day 0", "Non-Flu Day 7-9", "Non-Flu Day 80-100",
        "Standalone Baseline Visit", "Standalone Baseline Visit", "Standalone Baseline Visit"),
    sample.visitName = c(
        "Flu Year 1 Day 0", "Flu Year 1 Day 7", "Flu Year 1 Day 90",
        "Flu Year 2 Day 0", "Flu Year 2 Day 7", "Flu Year 2 Day 90",
        "Immune Variation Day 0", "Immune Variation Day 7", "Immune Variation Day 90",
        "Flu Year 1 Stand-Alone", "Flu Year 2 Stand-Alone", "Flu Year 3 Stand-Alone")
)

In [13]:
sample_meta <- sample_meta %>%
  left_join(timepoint_conversion)

Joining with `by = join_by(sample.visitName)`


In [14]:
nrow(sample_meta)

[1] 868

### Unperturbed whole blood samples

In [15]:
series_id <- "GSE279552"

Get sample metadata stored in GEO

In [16]:
sre <- suppressMessages(getGEO(series_id, GSEMatrix = FALSE))

In [17]:
sample_info <- GSMList(sre)

In [18]:
sample_characteristics <- map_dfr(
  sample_info,
  function(si) {
    raw_chars <- Meta(si)$characteristics_ch1
    char_names <- c("geo_accession", "source_name", "description", "title",
                    sub(":.+","", raw_chars))
    char_vals <- c(Meta(si)$geo_accession,
                   Meta(si)$source_name_ch1,
                   Meta(si)$description[length(Meta(si)$description)],
                   Meta(si)$title,
                   sub(".+: ","", raw_chars))
    names(char_vals) <- char_names
    as.data.frame(as.list(char_vals))
  }
)

Rename to prep for combination with sample_meta

In [19]:
sample_characteristics <- sample_characteristics %>%
  dplyr::select(description, geo_accession, title, donor, timepoint) %>%
  dplyr::rename(barcodes = description,
                subject.subjectGuid = donor,
                geo.accession = geo_accession,
                geo.title = title,
                geo.timepoint = timepoint)

In [20]:
head(sample_characteristics)

,barcodes,geo.accession,geo.title,subject.subjectGuid,geo.timepoint
,<chr>,<chr>,<chr>,<chr>,<chr>
1,lib80149,GSM8575086,BR2035 Y1 Flu Day 0,BR2035,Y1 Flu Day 0
2,lib80150,GSM8575087,BR2035 Y1 Flu Day 7-9,BR2035,Y1 Flu Day 7-9
3,lib80151,GSM8575088,BR2035 Y1 Flu Day 80-100,BR2035,Y1 Flu Day 80-100
4,lib80152,GSM8575089,BR2035 Non-Flu Day 0,BR2035,Non-Flu Day 0
5,lib80153,GSM8575090,BR2035 Non-Flu Day 7-9,BR2035,Non-Flu Day 7-9
6,lib80154,GSM8575091,BR2035 Non-Flu Day 80-100,BR2035,Non-Flu Day 80-100


In [21]:
nrow(sample_characteristics)

[1] 864

Combine and filter based on sample_meta

Subjects that aren't included in sample_meta (and cause NA values in sample.sampleKitGuid) were either not truly Healthy, or had limited sample collection:  
BR1020: No flu vaccine time points  
BR1034: Psoriasis  
BR1045: One sample is missing in sample_meta, but should be retained  
BR2007: Abnormal  
BR2049: Abnormal  


In [22]:
final_samples <- sample_characteristics %>%
  left_join(sample_meta, by = c("subject.subjectGuid","geo.timepoint")) %>%
  filter(!is.na(sample.sampleKitGuid) | subject.subjectGuid == "BR1045")

In [23]:
nrow(final_samples)

[1] 839

In [24]:
unstim_meta_csv <- paste0("output/sound-life_whole-blood_metadata_", Sys.Date(),".csv")

write.csv(
    final_samples,
    unstim_meta_csv,
    row.names = FALSE,
    quote = FALSE
)

In [25]:
supp_file <- getGEOSuppFiles(series_id)

Using locally cached version of supplementary file(s) GSE279552 found here:
/home/workspace/sound-life-scrna-analysis/04-file-sets/GSE279552/GSE279552_P462_genecounts.csv.gz 



In [26]:
mat <- read.csv(rownames(supp_file))

In [27]:
ensembl_ids <- mat$X
mat <- as.matrix(mat[,-1])
rownames(mat) <- ensembl_ids

In [28]:
sum(ensembl_ids %in% gene_meta$id)

[1] 52745

In [29]:
nrow(mat)

[1] 58302

In [30]:
missing <- setdiff(ensembl_ids, gene_meta$id)
length(missing)

[1] 5557

In [209]:
missing_mat <- mat[missing,]

In [210]:
missing_rsum <- rowSums(missing_mat)

In [211]:
max(missing_rsum)

[1] 56071

In [215]:
head(rownames(missing_mat))

[1] "ENSG00000263081" "ENSG00000263464" "ENSG00000268439" "ENSG00000271254"
[5] "ENSG00000272196" "ENSG00000273496"

In [212]:
rownames(missing_mat)[missing_rsum == max(missing_rsum)]

[1] "ENSG00000279396"

In [181]:
gene_meta %>%
  filter(symbol == "SOD2")

ERROR: [1m[33mError[39m in `filter()`:[22m
[1m[22m[36mℹ[39m In argument: `symbol == "SOD2"`.
[1mCaused by error:[22m
[33m![39m object 'symbol' not found


In [35]:
mat <- mat[,final_samples$barcodes]

In [36]:
str(mat)

 int [1:58302, 1:838] 1 0 91 39 18 2105 12 37 12 71 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:58302] "ENSG00000000003" "ENSG00000000005" "ENSG00000000419" "ENSG00000000457" ...
  ..$ : chr [1:838] "lib80149" "lib80150" "lib80151" "lib80152" ...


In [68]:
unstim_mat_csv <- paste0("output/sound-life_whole-blood_matrix_", Sys.Date(),".csv")

write.csv(
    mat,
    unstim_mat_csv,
    row.names = FALSE,
    quote = FALSE
)

In [71]:
unstim_adata <- anndata$AnnData(
    X = t(mat),
    obs = final_samples
)

In [72]:
unstim_adata

AnnData object with n_obs × n_vars = 985 × 58302
    obs: 'barcodes', 'geo_accession', 'title', 'subject.subjectGuid', 'timepoint', 'stimulation', 'cohort.cohortGuid', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'sample.sampleKitGuid', 'sample.visitName', 'sample.daysSinceFirstVisit', 'sample.drawYear'

### Perturbation whole blood samples

In [39]:
series_id <- "GSE279480"

In [40]:
sre <- suppressMessages(getGEO(series_id, GSEMatrix = FALSE))

In [41]:
sample_info <- GSMList(sre)

In [42]:
sample_characteristics <- map_dfr(
  sample_info,
  function(si) {
    raw_chars <- Meta(si)$characteristics_ch1
    char_names <- c("geo_accession", "source_name", "description", "title",
                    sub(":.+","", raw_chars))
    char_vals <- c(Meta(si)$geo_accession,
                   Meta(si)$source_name_ch1,
                   Meta(si)$description[length(Meta(si)$description)],
                   Meta(si)$title,
                   sub(".+: ","", raw_chars))
    names(char_vals) <- char_names
    as.data.frame(as.list(char_vals))
  }
)

In [43]:
sample_characteristics <- sample_characteristics %>%
  select(description, geo_accession, title, donor, timepoint, stimulation) %>%
  rename(barcodes = description) %>%
  rename(subject.subjectGuid = donor) %>%
  rename(geo.accession = geo_accession,
         geo.title = title,
         geo.timepoint = timepoint,
         geo.stimulation = stimulation)

In [44]:
head(sample_characteristics)

,barcodes,geo_accession,title,subject.subjectGuid,timepoint,stimulation
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,lib73151,GSM8572211,BR2031 LPS Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,LPS
2,lib73152,GSM8572212,BR2031 Null Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,Null
3,lib73153,GSM8572213,BR2031 Poly I:C Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,Poly I:C
4,lib73154,GSM8572214,BR2031 SEB Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,SEB
5,lib73163,GSM8572215,BR1003 Null Standalone Baseline Visit,BR1003,Standalone Baseline Visit,Null
6,lib73164,GSM8572216,BR1003 Poly I:C Standalone Baseline Visit,BR1003,Standalone Baseline Visit,Poly I:C


In [45]:
nrow(sample_characteristics)

[1] 1021

In [46]:
final_samples <- sample_characteristics %>%
  left_join(sample_meta, by = c("subject.subjectGuid","timepoint")) %>%
  filter(!is.na(sample.sampleKitGuid))

In [47]:
nrow(final_samples)

[1] 985

In [57]:
stim_meta_csv <- paste0("output/sound-life_whole-blood-stim_metadata_", Sys.Date(),".csv")

write.csv(
    final_samples,
    stim_meta_csv,
    row.names = FALSE,
    quote = FALSE
)

In [49]:
supp_file <- getGEOSuppFiles(series_id)

Using locally cached version of supplementary file(s) GSE279480 found here:
/home/workspace/sound-life-scrna-analysis/04-file-sets/GSE279480/GSE279480_P441_genecounts.csv.gz 



In [50]:
mat <- read.csv(rownames(supp_file))

In [51]:
uniprot_ids <- mat$X
mat <- as.matrix(mat[,-1])
rownames(mat) <- uniprot_ids

In [52]:
mat <- mat[,final_samples$barcodes]

In [53]:
str(mat)

 int [1:58302, 1:985] 7 0 182 34 1 2567 15 59 21 107 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:58302] "ENSG00000000003" "ENSG00000000005" "ENSG00000000419" "ENSG00000000457" ...
  ..$ : chr [1:985] "lib73151" "lib73152" "lib73153" "lib73154" ...


In [58]:
stim_mat_csv <- paste0("output/sound-life_whole-blood-stim_matrix_", Sys.Date(),".csv")

write.csv(
    mat,
    unstim_mat_csv,
    row.names = FALSE,
    quote = FALSE
)